# **MIMIC-IV Joins:**

## **Set-up:**

In [2]:
# Imports:
from pathlib import Path
import pandas as pd
import numpy as np

In [3]:
# Set up paths:
DATA_DIR = Path("/Users/gloriaduo/Documents/SJSU/SPRING 2026/CS 133 (Data Visualization)/Project/Predicting-Patient-Mortality/mimic-iv-3.1")
OUT_DIR = Path("/Users/gloriaduo/Documents/SJSU/SPRING 2026/CS 133 (Data Visualization)/Project/Predicting-Patient-Mortality/derived_results")
OUT_DIR.mkdir(exist_ok=True)

ICU_STAYS_CSV = DATA_DIR / "icu" / "icustays.csv"
PATIENTS_CSV = DATA_DIR / "hosp" / "patients.csv"
ADMISSIONS_CSV = DATA_DIR / "hosp" / "admissions.csv"

In [4]:
# int64 -> int32 downcast function:
def downcast_int64_to_int32(df):
    for col in df.columns:
        temp = df[col]

        # Only touch real int64 columns (strings, datetimes, floats, etc. are ignored).
        if temp.dtype != "int64":
            continue
        
        col_min = df[col].min()
        col_max = df[col].max()

        if (col_min >= np.iinfo(np.int32).min and col_max <= np.iinfo(np.int32).max):
            df[col] = df[col].astype(np.int32)
        else:
            print(f"Skipped {col} (out of int32 range).")

    return df

## **icustays.csv:**

In [5]:
# Load in icustays.csv (14 MB, safe to full load):
icustays = pd.read_csv(ICU_STAYS_CSV)

In [6]:
# Preview columns:
print("Columns for icustays.csv:\n\n", icustays.columns)

Columns for icustays.csv:

 Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los'],
      dtype='str')


In [7]:
# Preview description:
print("Description for icustays.csv:\n\n", icustays.describe())

Description for icustays.csv:

          subject_id       hadm_id       stay_id           los
count  9.445800e+04  9.445800e+04  9.445800e+04  94444.000000
mean   1.500422e+07  2.498185e+07  3.499832e+07      3.630025
std    2.884050e+06  2.884066e+06  2.886407e+06      5.402474
min    1.000003e+07  2.000009e+07  3.000015e+07      0.001250
25%    1.251463e+07  2.248212e+07  3.250678e+07      1.096212
50%    1.500554e+07  2.498248e+07  3.499944e+07      1.965648
75%    1.751758e+07  2.746506e+07  3.749099e+07      3.862575
max    1.999999e+07  2.999983e+07  3.999986e+07    226.403079


In [8]:
# Preview info:
print("Info for icustays.csv:\n\n", icustays.info())

<class 'pandas.DataFrame'>
RangeIndex: 94458 entries, 0 to 94457
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   subject_id      94458 non-null  int64  
 1   hadm_id         94458 non-null  int64  
 2   stay_id         94458 non-null  int64  
 3   first_careunit  94458 non-null  str    
 4   last_careunit   94458 non-null  str    
 5   intime          94458 non-null  str    
 6   outtime         94444 non-null  str    
 7   los             94444 non-null  float64
dtypes: float64(1), int64(3), str(4)
memory usage: 15.3 MB
Info for icustays.csv:

 None


In [9]:
# Preview data:
icustays.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


In [10]:
# Process dates for intime and outtime:
icustays["intime"] = pd.to_datetime(
    icustays["intime"], 
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

icustays["outtime"] = pd.to_datetime(
    icustays["outtime"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# Check that our datatype has changed:
icustays.info()

<class 'pandas.DataFrame'>
RangeIndex: 94458 entries, 0 to 94457
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   subject_id      94458 non-null  int64         
 1   hadm_id         94458 non-null  int64         
 2   stay_id         94458 non-null  int64         
 3   first_careunit  94458 non-null  str           
 4   last_careunit   94458 non-null  str           
 5   intime          94458 non-null  datetime64[us]
 6   outtime         94444 non-null  datetime64[us]
 7   los             94444 non-null  float64       
dtypes: datetime64[us](2), float64(1), int64(3), str(2)
memory usage: 11.9 MB


In [11]:
# Downcast:
downcast_icustays = downcast_int64_to_int32(icustays)

# View result:
downcast_icustays.info()

<class 'pandas.DataFrame'>
RangeIndex: 94458 entries, 0 to 94457
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   subject_id      94458 non-null  int32         
 1   hadm_id         94458 non-null  int32         
 2   stay_id         94458 non-null  int32         
 3   first_careunit  94458 non-null  str           
 4   last_careunit   94458 non-null  str           
 5   intime          94458 non-null  datetime64[us]
 6   outtime         94444 non-null  datetime64[us]
 7   los             94444 non-null  float64       
dtypes: datetime64[us](2), float64(1), int32(3), str(2)
memory usage: 10.8 MB


## **admissions.csv:**

In [12]:
# Load in admissions.csv (89.8 MB, safe to full load).
admissions = pd.read_csv(ADMISSIONS_CSV)

In [13]:
# Preview columns:
print("Columns for admissions.csv:\n\n", admissions.columns)

Columns for admissions.csv:

 Index(['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag'],
      dtype='str')


In [14]:
# Preview description:
print("Description for admissions.csv:\n\n", admissions.describe())

Description for admissions.csv:

          subject_id       hadm_id  hospital_expire_flag
count  5.460280e+05  5.460280e+05         546028.000000
mean   1.501118e+07  2.500100e+07              0.021612
std    2.877694e+06  2.888710e+06              0.145415
min    1.000003e+07  2.000002e+07              0.000000
25%    1.252380e+07  2.249662e+07              0.000000
50%    1.501961e+07  2.500385e+07              0.000000
75%    1.750403e+07  2.750282e+07              0.000000
max    1.999999e+07  2.999994e+07              1.000000


In [15]:
# Preview info:
print("Info for admissions.csv:\n\n", admissions.info())

<class 'pandas.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   subject_id            546028 non-null  int64
 1   hadm_id               546028 non-null  int64
 2   admittime             546028 non-null  str  
 3   dischtime             546028 non-null  str  
 4   deathtime             11790 non-null   str  
 5   admission_type        546028 non-null  str  
 6   admit_provider_id     546024 non-null  str  
 7   admission_location    546027 non-null  str  
 8   discharge_location    396210 non-null  str  
 9   insurance             536673 non-null  str  
 10  language              545253 non-null  str  
 11  marital_status        532409 non-null  str  
 12  race                  546028 non-null  str  
 13  edregtime             379240 non-null  str  
 14  edouttime             379240 non-null  str  
 15  hospital_expire_flag  546028 non-null  int64


In [16]:
# Preview data:
admissions.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0


In [17]:
# Process datetimes for admittime, dischtime, deathtime, edregtime, & edouttime.
time_features = ["admittime", "dischtime", "deathtime", "edregtime", "edouttime"]

for feature in time_features:
    admissions[feature] = pd.to_datetime(
    admissions[feature],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# Check that our datatypes have changed:
admissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int64         
 1   hadm_id               546028 non-null  int64         
 2   admittime             546028 non-null  datetime64[us]
 3   dischtime             546028 non-null  datetime64[us]
 4   deathtime             11790 non-null   datetime64[us]
 5   admission_type        546028 non-null  str           
 6   admit_provider_id     546024 non-null  str           
 7   admission_location    546027 non-null  str           
 8   discharge_location    396210 non-null  str           
 9   insurance             536673 non-null  str           
 10  language              545253 non-null  str           
 11  marital_status        532409 non-null  str           
 12  race                  546028 non-null  str           
 13  edregtime 

In [18]:
# Downcast:
downcast_admissions = downcast_int64_to_int32(admissions)

# View results:
downcast_admissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int32         
 1   hadm_id               546028 non-null  int32         
 2   admittime             546028 non-null  datetime64[us]
 3   dischtime             546028 non-null  datetime64[us]
 4   deathtime             11790 non-null   datetime64[us]
 5   admission_type        546028 non-null  str           
 6   admit_provider_id     546024 non-null  str           
 7   admission_location    546027 non-null  str           
 8   discharge_location    396210 non-null  str           
 9   insurance             536673 non-null  str           
 10  language              545253 non-null  str           
 11  marital_status        532409 non-null  str           
 12  race                  546028 non-null  str           
 13  edregtime 

In [19]:
# Convert hospital_expire_flag to boolean:
downcast_admissions["hospital_expire_flag"] = downcast_admissions["hospital_expire_flag"].astype(bool)

# View results:
downcast_admissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int32         
 1   hadm_id               546028 non-null  int32         
 2   admittime             546028 non-null  datetime64[us]
 3   dischtime             546028 non-null  datetime64[us]
 4   deathtime             11790 non-null   datetime64[us]
 5   admission_type        546028 non-null  str           
 6   admit_provider_id     546024 non-null  str           
 7   admission_location    546027 non-null  str           
 8   discharge_location    396210 non-null  str           
 9   insurance             536673 non-null  str           
 10  language              545253 non-null  str           
 11  marital_status        532409 non-null  str           
 12  race                  546028 non-null  str           
 13  edregtime 

In [20]:
# Based on documentation, admission_type only has 9 possibilities, so we can store as categorical.
# Double-check:
print("Number of unique values for admission_type:", downcast_admissions["admission_type"].nunique())
print("\nUnique categories for admission_type:", downcast_admissions["admission_type"].unique())

Number of unique values for admission_type: 9

Unique categories for admission_type: <ArrowStringArray>
[                     'URGENT',                    'EW EMER.',
              'EU OBSERVATION',           'OBSERVATION ADMIT',
 'SURGICAL SAME DAY ADMISSION',      'AMBULATORY OBSERVATION',
                'DIRECT EMER.',          'DIRECT OBSERVATION',
                    'ELECTIVE']
Length: 9, dtype: str


In [21]:
# Conversion for admission_type:
downcast_admissions["admission_type"] = downcast_admissions["admission_type"].astype("category")

# View results:
downcast_admissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int32         
 1   hadm_id               546028 non-null  int32         
 2   admittime             546028 non-null  datetime64[us]
 3   dischtime             546028 non-null  datetime64[us]
 4   deathtime             11790 non-null   datetime64[us]
 5   admission_type        546028 non-null  category      
 6   admit_provider_id     546024 non-null  str           
 7   admission_location    546027 non-null  str           
 8   discharge_location    396210 non-null  str           
 9   insurance             536673 non-null  str           
 10  language              545253 non-null  str           
 11  marital_status        532409 non-null  str           
 12  race                  546028 non-null  str           
 13  edregtime 

In [22]:
# admission_location and discharge_location are hospital operational categories. 
# They are neither numeric nor ordinal, so for modeling purposes, we should also convert them to categories.
# The documentation contains potential values, but let's print to double-check:
print("Number of unique values for admission_location:", downcast_admissions["admission_location"].nunique())
print("\nUnique categories for admission_location:", downcast_admissions["admission_location"].unique())
print("\nNumber of unique values for discharge_location:", downcast_admissions["discharge_location"].nunique())
print("\nUnique categories for discharge_location:", downcast_admissions["discharge_location"].unique())

Number of unique values for admission_location: 11

Unique categories for admission_location: <ArrowStringArray>
[                'TRANSFER FROM HOSPITAL',
                         'EMERGENCY ROOM',
                  'WALK-IN/SELF REFERRAL',
                     'PHYSICIAN REFERRAL',
                         'PROCEDURE SITE',
                        'CLINIC REFERRAL',
 'TRANSFER FROM SKILLED NURSING FACILITY',
                                   'PACU',
     'INTERNAL TRANSFER TO OR FROM PSYCH',
              'INFORMATION NOT AVAILABLE',
            'AMBULATORY SURGERY TRANSFER',
                                      nan]
Length: 12, dtype: str

Number of unique values for discharge_location: 13

Unique categories for discharge_location: <ArrowStringArray>
[                        'HOME',                      'HOSPICE',
                            nan,             'HOME HEALTH CARE',
     'SKILLED NURSING FACILITY',                        'REHAB',
                         'DIED', 'CHRON

In [23]:
# There are NaN values for both, so we can convert those to 'UNKNOWN'.
downcast_admissions["admission_location"] = (
    downcast_admissions["admission_location"]
    .astype("string")
    .str.strip()
    .fillna("UNKNOWN")
    .astype("category")
)

downcast_admissions["discharge_location"] = (
    downcast_admissions["discharge_location"]
    .astype("string")
    .str.strip()
    .fillna("UNKNOWN")
    .astype("category")
)

# View results:
downcast_admissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int32         
 1   hadm_id               546028 non-null  int32         
 2   admittime             546028 non-null  datetime64[us]
 3   dischtime             546028 non-null  datetime64[us]
 4   deathtime             11790 non-null   datetime64[us]
 5   admission_type        546028 non-null  category      
 6   admit_provider_id     546024 non-null  str           
 7   admission_location    546028 non-null  category      
 8   discharge_location    546028 non-null  category      
 9   insurance             536673 non-null  str           
 10  language              545253 non-null  str           
 11  marital_status        532409 non-null  str           
 12  race                  546028 non-null  str           
 13  edregtime 

In [24]:
# Moving on to insurance...
print("Number of unique values for insurance:", downcast_admissions["insurance"].nunique())
print("\nUnique values for insurance:", downcast_admissions["insurance"].unique())

Number of unique values for insurance: 5

Unique values for insurance: <ArrowStringArray>
['Medicaid', nan, 'Medicare', 'Private', 'Other', 'No charge']
Length: 6, dtype: str


In [25]:
# We can convert insurance to a category since there's low cardinality, and because the values are nominal:
downcast_admissions["insurance"] = (
    downcast_admissions["insurance"]
    .astype("string")
    .str.strip()
    .fillna("UNKNOWN")
    .astype("category")
)

# View results:
downcast_admissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int32         
 1   hadm_id               546028 non-null  int32         
 2   admittime             546028 non-null  datetime64[us]
 3   dischtime             546028 non-null  datetime64[us]
 4   deathtime             11790 non-null   datetime64[us]
 5   admission_type        546028 non-null  category      
 6   admit_provider_id     546024 non-null  str           
 7   admission_location    546028 non-null  category      
 8   discharge_location    546028 non-null  category      
 9   insurance             546028 non-null  category      
 10  language              545253 non-null  str           
 11  marital_status        532409 non-null  str           
 12  race                  546028 non-null  str           
 13  edregtime 

In [26]:
# Moving on to language...
print("Number of unique values for language:", downcast_admissions["language"].nunique())
print("\nUnique values for language:", downcast_admissions["language"].unique())

Number of unique values for language: 25

Unique values for language: <ArrowStringArray>
[               'English',                  'Other',                'Russian',
           'Kabuverdianu',                'Spanish', 'American Sign Language',
             'Vietnamese',             'Portuguese',                      nan,
   'Modern Greek (1453-)',                'Haitian',                'Persian',
                 'French',                'Chinese',                 'Arabic',
                  'Khmer',               'Japanese',                   'Thai',
                 'Polish',                'Italian',                'Amharic',
                 'Korean',                  'Hindi',                'Bengali',
               'Armenian',                 'Somali']
Length: 26, dtype: str


In [27]:
# Convert language to categorical:
downcast_admissions["language"] = (
    downcast_admissions["language"]
    .astype("string")
    .str.strip()
    .fillna("UNKNOWN")
    .astype("category")
)

# View results:
downcast_admissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int32         
 1   hadm_id               546028 non-null  int32         
 2   admittime             546028 non-null  datetime64[us]
 3   dischtime             546028 non-null  datetime64[us]
 4   deathtime             11790 non-null   datetime64[us]
 5   admission_type        546028 non-null  category      
 6   admit_provider_id     546024 non-null  str           
 7   admission_location    546028 non-null  category      
 8   discharge_location    546028 non-null  category      
 9   insurance             546028 non-null  category      
 10  language              546028 non-null  category      
 11  marital_status        532409 non-null  str           
 12  race                  546028 non-null  str           
 13  edregtime 

In [28]:
# Moving on to marital_status...
print("Number of unique values for marital_status:", downcast_admissions["marital_status"].nunique())
print("\nUnique values for marital_status:", downcast_admissions["marital_status"].unique())

Number of unique values for marital_status: 4

Unique values for marital_status: <ArrowStringArray>
['WIDOWED', 'SINGLE', 'MARRIED', 'DIVORCED', nan]
Length: 5, dtype: str


In [29]:
# Convert marital_status to categorical:
downcast_admissions["marital_status"] = (
    downcast_admissions["marital_status"]
    .astype("string")
    .str.strip()
    .fillna("UNKNOWN")
    .astype("category")
)

# View results:
downcast_admissions.info()

<class 'pandas.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int32         
 1   hadm_id               546028 non-null  int32         
 2   admittime             546028 non-null  datetime64[us]
 3   dischtime             546028 non-null  datetime64[us]
 4   deathtime             11790 non-null   datetime64[us]
 5   admission_type        546028 non-null  category      
 6   admit_provider_id     546024 non-null  str           
 7   admission_location    546028 non-null  category      
 8   discharge_location    546028 non-null  category      
 9   insurance             546028 non-null  category      
 10  language              546028 non-null  category      
 11  marital_status        546028 non-null  category      
 12  race                  546028 non-null  str           
 13  edregtime 

In [30]:
# Moving on to race...
print("Number of unique values for race:", downcast_admissions["race"].nunique())
print("\nUnique values for race:", downcast_admissions["race"].unique())

Number of unique values for race: 33

Unique values for race: <ArrowStringArray>
[                                    'WHITE',
                                     'OTHER',
                    'BLACK/AFRICAN AMERICAN',
                          'UNABLE TO OBTAIN',
                                   'UNKNOWN',
                           'WHITE - RUSSIAN',
                        'BLACK/CAPE VERDEAN',
 'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER',
                                'PORTUGUESE',
                    'WHITE - OTHER EUROPEAN',
            'HISPANIC/LATINO - PUERTO RICAN',
                                     'ASIAN',
                           'ASIAN - CHINESE',
               'HISPANIC/LATINO - DOMINICAN',
              'HISPANIC/LATINO - SALVADORAN',
                             'BLACK/AFRICAN',
              'HISPANIC/LATINO - GUATEMALAN',
                  'ASIAN - SOUTH EAST ASIAN',
                         'WHITE - BRAZILIAN',
                            'SOUTH AMERICAN',

In [31]:
# Since there are clearly a lot of subcategories for race, 
# I'm going to collapse them into the standard OMB-style race groups based on U.S. Census Bureau criteria:
# https://www.census.gov/about/our-research/race-ethnicity/standards-updates.html#:~:text=The%20updated%20standards%20have%20seven%20co-equal%20minimum%20categories,African%2C%20Native%20Hawaiian%20or%20Pacific%20Islander%2C%20and%20White.
def collapse_race(race):
    if pd.isna(race):
        return "UNKNOWN"

    race = race.strip().upper()

    # Multiple.
    if "MULTIPLE" in race:
        return "MULTIPLE"

    # Unknown.
    if "DECLINED" in race or "UNABLE" in race or "UNKNOWN" in race:
        return "UNKNOWN"

    # Hispanic/Latino.
    if "HISPANIC" in race or "LATINO" in race:
        return "HISPANIC/LATINO"

    # American Indian/Alaska Native.
    if "AMERICAN INDIAN" in race or "ALASKA NATIVE" in race:
        return "AMERICAN INDIAN/ALASKA NATIVE"

    # Pacific Islander/Native Hawaiian.
    if "PACIFIC" in race or "HAWAIIAN" in race:
        return "NATIVE HAWAIIAN/PACIFIC ISLANDER"

    # Asian.
    if "ASIAN" in race:
        return "ASIAN"

    # Black.
    if "BLACK" in race:
        return "BLACK"

    # MENA (skipped this one because the category labels are not present in dataset.

    # White.
    if "WHITE" in race or "PORTUGUESE" in race:
        return "WHITE"

    # Other.
    else:
        return "OTHER"

# Update dataframe with new feature.
downcast_admissions["race_grouped"] = downcast_admissions["race"].apply(collapse_race).astype("category")
downcast_admissions.drop(columns=["race"], inplace=True)

# View results:
downcast_admissions.info()
print("\nNumber of unique values for race:", downcast_admissions["race_grouped"].nunique())
print("\nUnique values for race:", downcast_admissions["race_grouped"].unique())

<class 'pandas.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int32         
 1   hadm_id               546028 non-null  int32         
 2   admittime             546028 non-null  datetime64[us]
 3   dischtime             546028 non-null  datetime64[us]
 4   deathtime             11790 non-null   datetime64[us]
 5   admission_type        546028 non-null  category      
 6   admit_provider_id     546024 non-null  str           
 7   admission_location    546028 non-null  category      
 8   discharge_location    546028 non-null  category      
 9   insurance             546028 non-null  category      
 10  language              546028 non-null  category      
 11  marital_status        546028 non-null  category      
 12  edregtime             379240 non-null  datetime64[us]
 13  edouttime 

## **patients.csv:**

In [32]:
# Load in patients.csv (11.5 MB, safe to full load):
patients = pd.read_csv(PATIENTS_CSV)

In [33]:
# Preview columns:
print("Columns for patients.csv:\n\n", patients.columns)

Columns for patients.csv:

 Index(['subject_id', 'gender', 'anchor_age', 'anchor_year',
       'anchor_year_group', 'dod'],
      dtype='str')


In [34]:
# Preview description:
print("Description for patients.csv:\n\n", patients.describe())

Description for patients.csv:

          subject_id     anchor_age    anchor_year
count  3.646270e+05  364627.000000  364627.000000
mean   1.501166e+07      48.875097    2150.835404
std    2.885013e+06      20.943316      23.395666
min    1.000003e+07      18.000000    2110.000000
25%    1.251201e+07      29.000000    2131.000000
50%    1.501868e+07      48.000000    2151.000000
75%    1.750900e+07      65.000000    2171.000000
max    1.999999e+07      91.000000    2208.000000


In [35]:
# Preview info:
print("Info for patients.csv:\n\n", patients.info())

<class 'pandas.DataFrame'>
RangeIndex: 364627 entries, 0 to 364626
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   subject_id         364627 non-null  int64
 1   gender             364627 non-null  str  
 2   anchor_age         364627 non-null  int64
 3   anchor_year        364627 non-null  int64
 4   anchor_year_group  364627 non-null  str  
 5   dod                38301 non-null   str  
dtypes: int64(3), str(3)
memory usage: 21.3 MB
Info for patients.csv:

 None


In [36]:
# Preview data:
patients.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


In [37]:
# Downcast:
downcast_patients = downcast_int64_to_int32(patients)

# View results:
downcast_patients.info()

<class 'pandas.DataFrame'>
RangeIndex: 364627 entries, 0 to 364626
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   subject_id         364627 non-null  int32
 1   gender             364627 non-null  str  
 2   anchor_age         364627 non-null  int32
 3   anchor_year        364627 non-null  int32
 4   anchor_year_group  364627 non-null  str  
 5   dod                38301 non-null   str  
dtypes: int32(3), str(3)
memory usage: 17.1 MB


In [38]:
# In our dataset, gender is a binary variable:
print("Number of unique values for gender:", downcast_patients["gender"].nunique())
print("\nUnique values for gender:", downcast_patients["gender"].unique())

Number of unique values for gender: 2

Unique values for gender: <ArrowStringArray>
['F', 'M']
Length: 2, dtype: str


In [39]:
# Continue with conversion to bool (no missing values):
downcast_patients["is_male"] = (
    downcast_patients["gender"] == "M" # "M" -> True, "F" -> False
)

# Drop gender:
downcast_patients.drop(columns=["gender"], inplace=True)

# View results:
downcast_patients.info()

<class 'pandas.DataFrame'>
RangeIndex: 364627 entries, 0 to 364626
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   subject_id         364627 non-null  int32
 1   anchor_age         364627 non-null  int32
 2   anchor_year        364627 non-null  int32
 3   anchor_year_group  364627 non-null  str  
 4   dod                38301 non-null   str  
 5   is_male            364627 non-null  bool 
dtypes: bool(1), int32(3), str(2)
memory usage: 14.3 MB


In [40]:
# Convert dod to datetime:
downcast_patients["dod"] = pd.to_datetime(downcast_patients["dod"], errors="coerce")

# View results:
downcast_patients.info()

<class 'pandas.DataFrame'>
RangeIndex: 364627 entries, 0 to 364626
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   subject_id         364627 non-null  int32         
 1   anchor_age         364627 non-null  int32         
 2   anchor_year        364627 non-null  int32         
 3   anchor_year_group  364627 non-null  str           
 4   dod                38301 non-null   datetime64[us]
 5   is_male            364627 non-null  bool          
dtypes: bool(1), datetime64[us](1), int32(3), str(1)
memory usage: 13.9 MB


## **Build Anchor Cohort:**

Our cohort is ultimately made up of:

  - Patients who are transferred to the ICU.
  - Patients who are either 18 or older.
  - The unit/index will be stay_id (unique to ICU transfer).

Since our base cohort uses stay_id as the index, we start joining from **downcast_icustays**. This guarantees that there is only one row per ICU stay.

In [41]:
# Create base cohort:
cohort = icustays.copy()

# View results:
cohort.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


We need information that is known at the time of ICU transfer, which includes static hospital admission information. This means we need to join on **downcast_admissions**.

In [42]:
# Join icustays + admissions on hadm_id and subject_id.
cohort = cohort.merge(
    downcast_admissions,
    on=["subject_id", "hadm_id"],
    how="left",
    validate="many_to_one"
)

# View results:
cohort.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,...,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,edregtime,edouttime,hospital_expire_flag,race_grouped
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,2180-07-23 12:35:00,2180-07-25 17:55:00,...,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,2180-07-23 05:54:00,2180-07-23 14:00:00,False,WHITE
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,2150-11-02 18:02:00,2150-11-12 13:45:00,...,P26QQ4,EMERGENCY ROOM,REHAB,Medicare,English,WIDOWED,2150-11-02 11:41:00,2150-11-02 19:37:00,False,WHITE
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,2189-06-27 07:38:00,2189-07-03 03:00:00,...,P06OTX,EMERGENCY ROOM,HOME HEALTH CARE,Medicare,English,MARRIED,2189-06-27 06:25:00,2189-06-27 08:42:00,False,BLACK
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,2157-11-18 22:56:00,2157-11-25 18:00:00,...,P3610N,EMERGENCY ROOM,HOME HEALTH CARE,Private,Other,MARRIED,2157-11-18 17:38:00,2157-11-19 01:24:00,False,WHITE
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,2157-12-18 16:58:00,2157-12-24 14:55:00,...,P276OU,PHYSICIAN REFERRAL,HOME HEALTH CARE,Private,Other,MARRIED,NaT,NaT,False,WHITE


We also need to join on **downcast_patients** for some patient-specific information.

In [43]:
# Join cohort on subject_id:
cohort = cohort.merge(
    downcast_patients,
    on="subject_id",
    how="left",
    validate="many_to_one"
)

# View results:
cohort.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,...,marital_status,edregtime,edouttime,hospital_expire_flag,race_grouped,anchor_age,anchor_year,anchor_year_group,dod,is_male
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,2180-07-23 12:35:00,2180-07-25 17:55:00,...,WIDOWED,2180-07-23 05:54:00,2180-07-23 14:00:00,False,WHITE,52,2180,2014 - 2016,2180-09-09,False
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,2150-11-02 18:02:00,2150-11-12 13:45:00,...,WIDOWED,2150-11-02 11:41:00,2150-11-02 19:37:00,False,WHITE,86,2150,2008 - 2010,2152-01-30,False
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,2189-06-27 07:38:00,2189-07-03 03:00:00,...,MARRIED,2189-06-27 06:25:00,2189-06-27 08:42:00,False,BLACK,73,2186,2008 - 2010,2193-08-26,False
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,2157-11-18 22:56:00,2157-11-25 18:00:00,...,MARRIED,2157-11-18 17:38:00,2157-11-19 01:24:00,False,WHITE,55,2157,2011 - 2013,NaT,False
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,2157-12-18 16:58:00,2157-12-24 14:55:00,...,MARRIED,NaT,NaT,False,WHITE,55,2157,2011 - 2013,NaT,False


We need to filter out ICU patients who are under 18, if there are any.

In [44]:
# Filter out patients under 18:
cohort["admission_year"] = cohort["admittime"].dt.year

cohort["age"] = (
    cohort["anchor_age"] +
    (cohort["admission_year"] - cohort["anchor_year"])
)

cohort = cohort[cohort["age"] >= 18]

# View results:
cohort.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,...,edouttime,hospital_expire_flag,race_grouped,anchor_age,anchor_year,anchor_year_group,dod,is_male,admission_year,age
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,2180-07-23 12:35:00,2180-07-25 17:55:00,...,2180-07-23 14:00:00,False,WHITE,52,2180,2014 - 2016,2180-09-09,False,2180,52
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,2150-11-02 18:02:00,2150-11-12 13:45:00,...,2150-11-02 19:37:00,False,WHITE,86,2150,2008 - 2010,2152-01-30,False,2150,86
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,2189-06-27 07:38:00,2189-07-03 03:00:00,...,2189-06-27 08:42:00,False,BLACK,73,2186,2008 - 2010,2193-08-26,False,2189,76
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,2157-11-18 22:56:00,2157-11-25 18:00:00,...,2157-11-19 01:24:00,False,WHITE,55,2157,2011 - 2013,NaT,False,2157,55
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,2157-12-18 16:58:00,2157-12-24 14:55:00,...,NaT,False,WHITE,55,2157,2011 - 2013,NaT,False,2157,55


We need to handle the fact that some of our patients are missing an outtime (this can happen because they died in the ICU, or due to some other unforeseen issues with administrative records). 

In [45]:
# Calculate the number of missing values for outtime:
missing_outtime = cohort[cohort["outtime"].isna()]
print("Number of ICU stays missing an outtime:", missing_outtime.shape[0])

Number of ICU stays missing an outtime: 14


In [46]:
# Calculate the number of rows where missing outtime has a deathtime:
missing_outtime["deathtime"].notna().mean()

np.float64(0.5)

In [47]:
# Based on our investigation, it appears that just because the outtime is missing doesn't mean the patient died in the ICU. 
# Since we have no clear way of knowing, I'll just drop the 14 rows.
cohort = cohort.dropna(subset=["outtime"])

In [48]:
# Now we need to figure out if a patient died in the ICU or after they were discharged (deathtime is from hosp module):
cohort["ICU_mortality"] = (
    cohort["deathtime"].notna()
    & (cohort["deathtime"] >= cohort["intime"])
    & (cohort["deathtime"] <= cohort["outtime"])
)

# View results:
cohort.info()

<class 'pandas.DataFrame'>
Index: 94444 entries, 0 to 94457
Data columns (total 30 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   subject_id            94444 non-null  int32         
 1   hadm_id               94444 non-null  int32         
 2   stay_id               94444 non-null  int32         
 3   first_careunit        94444 non-null  str           
 4   last_careunit         94444 non-null  str           
 5   intime                94444 non-null  datetime64[us]
 6   outtime               94444 non-null  datetime64[us]
 7   los                   94444 non-null  float64       
 8   admittime             94444 non-null  datetime64[us]
 9   dischtime             94444 non-null  datetime64[us]
 10  deathtime             11334 non-null  datetime64[us]
 11  admission_type        94444 non-null  category      
 12  admit_provider_id     94442 non-null  str           
 13  admission_location    94444 non-

In [49]:
# Check how many deaths we have to work with:
cohort["ICU_mortality"].sum()

np.int64(7329)

Now we can handle ICU time-to-event (the event being death or discharge).

In [50]:
# If a patient died in the ICU, use their deathtime. Otherwise, use their outtime.
cohort["ICU_endtime"] = cohort["outtime"]
cohort.loc[cohort["ICU_mortality"], "ICU_endtime"] = cohort.loc[cohort["ICU_mortality"], "deathtime"]

# Calculate time to event:
cohort["ICU_time_to_event_hours"] = (
    (cohort["ICU_endtime"] - cohort["intime"])
    .dt.total_seconds() / 3600
)

# View results:
cohort.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,admittime,dischtime,...,anchor_age,anchor_year,anchor_year_group,dod,is_male,admission_year,age,ICU_mortality,ICU_endtime,ICU_time_to_event_hours
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,2180-07-23 12:35:00,2180-07-25 17:55:00,...,52,2180,2014 - 2016,2180-09-09,False,2180,52,False,2180-07-23 23:50:47,9.846389
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,2150-11-02 18:02:00,2150-11-12 13:45:00,...,86,2150,2008 - 2010,2152-01-30,False,2150,86,False,2150-11-06 17:03:17,93.438056
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,2189-06-27 07:38:00,2189-07-03 03:00:00,...,73,2186,2008 - 2010,2193-08-26,False,2189,76,False,2189-06-27 20:38:27,11.940833
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,2157-11-18 22:56:00,2157-11-25 18:00:00,...,55,2157,2011 - 2013,NaT,False,2157,55,False,2157-11-21 22:08:00,26.832778
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,2157-12-18 16:58:00,2157-12-24 14:55:00,...,55,2157,2011 - 2013,NaT,False,2157,55,False,2157-12-20 14:27:41,22.754722


In [51]:
# Let's do some checks before proceeding...

# Check our ICU mortality percentage/proportion:
print("Proportion of ICU stays that ended in death:", cohort["ICU_mortality"].mean())

# Check our ICU stay duration distribution:
cohort["ICU_time_to_event_hours"].describe(percentiles=[.25, .5, .75, .9, .95])

Proportion of ICU stays that ended in death: 0.07760154165431367


count    94444.000000
mean        86.799918
std        129.546990
min          0.030000
25%         26.156319
50%         46.964028
75%         92.433958
90%        189.624000
95%        303.626917
max       5433.673889
Name: ICU_time_to_event_hours, dtype: float64

I want to do some pruning before our checkpoint (some of these features are unimportant for our modeling purposes).

In [52]:
# Inspect columns: 
cohort.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status',
       'edregtime', 'edouttime', 'hospital_expire_flag', 'race_grouped',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'dod', 'is_male',
       'admission_year', 'age', 'ICU_mortality', 'ICU_endtime',
       'ICU_time_to_event_hours'],
      dtype='str')

In [53]:
# Define the columns that we want:
baseline_cols = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "first_careunit",
    "intime", 
    "outtime",
    "admission_type",
    "admission_location",
    "insurance",
    "language",
    "marital_status",
    "race_grouped",
    "is_male",
    "age",
    "ICU_mortality",
    "ICU_endtime",
    "ICU_time_to_event_hours"
]

In [54]:
# Drop other features:
cohort = cohort[baseline_cols].copy()

## **Checkpoint:**

In [55]:
# Saving results up until this point:
import pyarrow as pa
import pyarrow.parquet as pq

# Cohort:
cohort.to_parquet(
    "/Users/gloriaduo/Documents/SJSU/SPRING 2026/CS 133 (Data Visualization)/Project/Predicting-Patient-Mortality/derived_results/baseline_cohort.parquet",
    index=False
)

## **Add Time-Series Data:**

### **Baseline (24-hour ICU window ONLY):**

For our 24-hour baseline, we only want to use measurements that fall into the following time period:

  - **intime <= charttime <= intime + 24 hours**

In [56]:
# Define the window end (window start is intime, which is already in our baseline cohort):
cohort["window_24h_end"] = cohort["intime"] + pd.Timedelta(hours=24)

Next, we need to decide on what features to include. For this section, since I'm lacking a lot of domain knowledge, I had ChatGPT analyze the feature set and decide on a minimum baseline feature set. We can edit this going forward if we want to capture more features. The features that it strongly suggested including are:

### **Vitals** ('chartevents', which contains all the charted data available for a patient during their ICU stay):
    - Heart rate
    - Mean arterial pressure
    - Respiratory rate
    - SpO2
    - Temperature
    - GCS                                                                                                                                                   
### **Labs** ('labevents', which stores the results of all laboratory measurements made for a single patient):
    - Lactate
    - Creatinine
    - WBC
    - Platelets
    - Sodium
    - Potassium
    - Bilirubin

Moving on, we need to correctly identify itemids from 'd_items'.

In [57]:
# Load in d_items (366.4 KB, safe to full-load):
D_ITEMS_CSV = DATA_DIR / "icu" / "d_items.csv"
d_items = pd.read_csv(D_ITEMS_CSV)

# Preview contents:
d_items.head()

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,220001,Problem List,Problem List,chartevents,General,NaN,Text,NaN,NaN
1,220003,ICU Admission date,ICU Admission date,datetimeevents,ADT,NaN,Date and time,NaN,NaN
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
3,220046,Heart rate Alarm - High,HR Alarm - High,chartevents,Alarms,bpm,Numeric,NaN,NaN
4,220047,Heart Rate Alarm - Low,HR Alarm - Low,chartevents,Alarms,bpm,Numeric,NaN,NaN


In [58]:
# View columns:
d_items.columns

Index(['itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname',
       'param_type', 'lownormalvalue', 'highnormalvalue'],
      dtype='str')

Notice that there are non-numeric values for the param_type feature. For a basic pipeline, we'll only use the numeric values since they're easiest to process for modeling. We can expand on this later too if we have time.

In [59]:
# Only select rows that link to 'chartevents' and are numeric:
chart_items = d_items[
    (d_items["linksto"] == "chartevents") &
    (d_items["param_type"] == "Numeric")
].copy()

In [60]:
# Inspect what we collected in the chart_items rows:
chart_items["category"].value_counts()

category
Labs                           118
Respiratory                     84
Skin - Impairment               60
Scores - APACHE IV (2)          46
Scores - APACHE II              34
Routine Vital Signs             33
Hemodynamics                    31
ECMO                            31
Alarms                          29
Dialysis                        25
Impella                         23
Pain/Sedation                   19
OT Notes                        18
PA Line Insertion               17
Cardiovascular (Pacer Data)     15
NICOM                           13
Durable VAD                     12
IABP                            11
PiCCO                           10
General                          9
Intubation                       9
Treatments                       8
Centrimag                        8
GI/GU                            7
Tandem Heart                     7
Neurological                     7
Heartware                        6
ZIntake                          5
Thoracentes

Note that d_items is just a dictionary of sorts to see how many different labels fall under each category. We can examine this more closely.

In [61]:
# Examine different candidate labels for the Routine Vital Signs category:
chart_items.loc[
    chart_items["category"] == "Routine Vital Signs",
    ["itemid", "label", "unitname", "param_type"]
].sort_values("label")

,itemid,label,unitname,param_type
1213,225310,ART BP Diastolic,mmHg,Numeric
1214,225312,ART BP Mean,mmHg,Numeric
1212,225309,ART BP Systolic,mmHg,Numeric
2333,227632,Arctic Sun/Alsius Temp #1 C,°C,Numeric
2334,227634,Arctic Sun/Alsius Temp #2 C,°C,Numeric
7,220051,Arterial Blood Pressure diastolic,mmHg,Numeric
8,220052,Arterial Blood Pressure mean,mmHg,Numeric
6,220050,Arterial Blood Pressure systolic,mmHg,Numeric
339,223763,Bladder Pressure,mmHg,Numeric
1814,226329,Blood Temperature CCO (C),°C,Numeric


Once again, I'm lacking domain knowledge, so I let ChatGPT decide! We'll be using the following vital signs:

  - 220045 Heart Rate
  - 220052 Arterial Blood Pressure mean
  - 220181 Non Invasive Blood Pressure mean
  - 223762 Temperature Celsius

In [62]:
# Examine different candidate labels for the Respiratory category:
chart_items.loc[chart_items["category"] == "Respiratory",
    ["itemid", "label", "unitname", "param_type"]
].sort_values("label")

,itemid,label,unitname,param_type
3770,229659,% Minute Volume,NaN,Numeric
812,224704,ATC %,%,Numeric
400,223876,Apnea Interval,sec,Numeric
2285,227579,BiPap EPAP,cmH2O,Numeric
2286,227580,BiPap IPAP,cmH2O,Numeric
...,...,...,...,...
680,224420,Vital Cap,Liters,Numeric
29,220218,Vital Capacity,Liters,Numeric
398,223874,Vti High,mL,Numeric
811,224703,ZATC Tube #,NaN,Numeric


In [63]:
# Inspect Respiratory labels more closely (truncated result above):
chart_items.loc[chart_items["category"] == "Respiratory", "label"].unique()

<ArrowStringArray>
[                    'Respiratory Rate',
                       'Vital Capacity',
                       'CO2 production',
          'O2 saturation pulseoxymetry',
                           'Resistance',
                             'PEEP set',
                              'O2 Flow',
                 'Inspired O2 Fraction',
                   'Inspired Gas Temp.',
                             'Paw High',
                             'Vti High',
                            'Fspn High',
                       'Apnea Interval',
                          'MDI #1 Puff',
                          'MDI #2 Puff',
                          'MDI #3 Puff',
             'Small Volume Neb Dose #2',
                        'Cuff Pressure',
                     'Cuff Volume (mL)',
                 'Negative Insp. Force',
                            'Vital Cap',
                             'Spont Vt',
                             'Spont RR',
                   'Tidal Volume (set)

In [64]:
# Grab Respiratory Rate & O2 saturation pulseoxymetry's itemids:
print(chart_items.loc[chart_items["label"] == "Respiratory Rate", "itemid"])
print(chart_items.loc[chart_items["label"] == "O2 saturation pulseoxymetry", "itemid"])

28    220210
Name: itemid, dtype: int64
36    220277
Name: itemid, dtype: int64


In [65]:
# Group our selected features:
vital_itemids = {
    "heart_rate": [220045],
    "map": [220052, 220181],
    "temperature": [223762],
    "resp_rate": [220210],
    "spo2": [220277]
}

# Flatten:
selected_itemids = [220045, 220052, 220181, 223762, 220210, 220277]

# Verify that we selected the correct items before moving on:
chart_items[chart_items["itemid"].isin(selected_itemids)][
    ["itemid", "label", "unitname"]
]

,itemid,label,unitname
2,220045,Heart Rate,bpm
8,220052,Arterial Blood Pressure mean,mmHg
26,220181,Non Invasive Blood Pressure mean,mmHg
28,220210,Respiratory Rate,insp/min
36,220277,O2 saturation pulseoxymetry,%
338,223762,Temperature Celsius,°C


Now we identify lab itemids from 'd_labitems'. This is very similar to what we just did for 'itemids'.

In [66]:
# Load in labevents (63.1 KB, safe to full-load):
D_LABITEMS_CSV = DATA_DIR / "hosp" / "d_labitems.csv"
d_labitems = pd.read_csv(D_LABITEMS_CSV)

# Preview contents:
d_labitems.head()

,itemid,label,fluid,category
0,50801,Alveolar-arterial Gradient,Blood,Blood Gas
1,50802,Base Excess,Blood,Blood Gas
2,50803,"Calculated Bicarbonate, Whole Blood",Blood,Blood Gas
3,50804,Calculated Total CO2,Blood,Blood Gas
4,50805,Carboxyhemoglobin,Blood,Blood Gas


In [67]:
# Explore the categories:
d_labitems["category"].value_counts()

category
Chemistry     800
Hematology    785
Blood Gas      65
Name: count, dtype: int64

In [68]:
# Now examine the labels:
d_labitems["label"].unique()

<ArrowStringArray>
[         'Alveolar-arterial Gradient',                         'Base Excess',
 'Calculated Bicarbonate, Whole Blood',                'Calculated Total CO2',
                   'Carboxyhemoglobin',               'Chloride, Whole Blood',
                        'Free Calcium',                             'Glucose',
              'Hematocrit, Calculated',                          'Hemoglobin',
 ...
                             'RdRP Ct',                          'RdRP Endpt',
         'Helicobacter pylori Antigen',       'Estimated GFR (CKD- EPI 2021)',
                            'CBC Only',                       '% Parasitemia',
                     'Cytospin Review',                'IPT (Clinical Trial)',
                                 'PAN',                                'MXD%']
Length: 1192, dtype: str

Ok clearly there are too many unique values. Let's see if we can just parse them from the labels column.

In [69]:
# Lactate:
d_labitems[d_labitems["label"].str.contains("lactate", case=False, na=False)]

,itemid,label,fluid,category
11,50813,Lactate,Blood,Blood Gas
41,50843,"Lactate Dehydrogenase, Ascites",Ascites,Chemistry
151,50954,Lactate Dehydrogenase (LD),Blood,Chemistry
242,51054,"Lactate Dehydrogenase, Pleural",Pleural,Chemistry
911,51795,"Lactate Dehydrogenase, CSF",Cerebrospinal Fluid,Chemistry
1035,51944,"Lactate Dehydrogenase, Stool",Stool,Chemistry
1516,52442,Lactate,Blood,Blood Gas
1615,53154,Lactate,Blood,Chemistry


Interesting. There are different Lactate labels. After a little bit of research, it appears that lactate-measuring labs can be conducted either through arterial blood gas panels or through standard metabolic panels. Since they're all measuring the same thing, we'll use all the associated itemids.

In [70]:
# Grab lactate itemids:
lactate_ids = [50813, 52442, 53154]

In [71]:
# Creatinine:
d_labitems[d_labitems["label"].str.contains("creatinine", case=False, na=False)]

,itemid,label,fluid,category
39,50841,"Creatinine, Ascites",Ascites,Chemistry
110,50912,Creatinine,Blood,Chemistry
209,51021,"Creatinine, Joint Fluid",Joint Fluid,Chemistry
220,51032,"Creatinine, Body Fluid",Other Body Fluid,Chemistry
240,51052,"Creatinine, Pleural",Pleural,Chemistry
255,51067,24 hr Creatinine,Urine,Chemistry
258,51070,"Albumin/Creatinine, Urine",Urine,Chemistry
261,51073,"Amylase/Creatinine Ratio, Urine",Urine,Chemistry
268,51080,Creatinine Clearance,Urine,Chemistry
269,51081,"Creatinine, Serum",Urine,Chemistry


There are multiple creatinine labels, but we specifically want to focus on serum creatinine (i.e., creatinine measurements taken from blood). This is because urine creatinine measurements are usually used to normalize other kidney function markers. Furthermore, serum creatinine concentrations reflect kidney filtration ability (what the kidneys FAILED to filter), whereas urine creatinine concentrations reflect what the kidneys did filter out, but as we mentioned above, varies with patient demographic + hydration levels. So serum creatinine concentrations are a more direct measurement.

In [72]:
# Grab serum creatinine itemids:
creatinine_ids = [50912, 52024, 52546]

In [73]:
# White blood cells:
d_labitems[d_labitems["label"].str.contains("white blood", case=False, na=False)]

,itemid,label,fluid,category
486,51301,White Blood Cells,Blood,Hematology
872,51755,White Blood Cells,Blood,Chemistry
873,51756,White Blood Cells,Blood,Chemistry


In [74]:
# Grab WBC itemids:
wbc_ids = [51301, 51755, 51756]

In [75]:
# Platelets:
d_labitems[d_labitems["label"].str.contains("platelet", case=False, na=False)]

,itemid,label,fluid,category
425,51240,Large Platelets,Blood,Hematology
449,51264,Platelet Clumps,Blood,Hematology
450,51265,Platelet Count,Blood,Hematology
451,51266,Platelet Smear,Blood,Hematology
1195,52105,Direct Antiplatelet Antibodies,Blood,Hematology
1231,52142,Mean Platelet Volume,Blood,Hematology
1248,52159,Platelet Aggregation,Blood,Hematology
1648,53189,Platelet Count,Blood,Chemistry


In [76]:
# Grab platelet count itemids:
platelet_ids = [51265, 53189]

In [77]:
# Sodium:
d_labitems[d_labitems["label"].str.contains("sodium", case=False, na=False)]

,itemid,label,fluid,category
22,50824,"Sodium, Whole Blood",Blood,Blood Gas
32,50834,"Sodium, Body Fluid",Other Body Fluid,Blood Gas
46,50848,"Sodium, Ascites",Ascites,Chemistry
180,50983,Sodium,Blood,Chemistry
230,51042,"Sodium, Body Fluid",Other Body Fluid,Chemistry
246,51058,"Sodium, Pleural",Pleural,Chemistry
253,51065,"Sodium, Stool",Stool,Chemistry
288,51100,"Sodium, Urine",Urine,Chemistry
917,51801,"Sodium, CSF",Cerebrospinal Fluid,Chemistry
939,51823,"Sodium, Joint Fluid",Joint Fluid,Chemistry


Pick the sodium labels that reflect systemic electrolyte levels (blood as fluid).

In [78]:
# Grab sodium itemids:
sodium_ids = [50824, 50983, 52455, 52623]

In [79]:
# Potassium:
d_labitems[d_labitems["label"].str.contains("potassium", case=False, na=False)]

,itemid,label,fluid,category
20,50822,"Potassium, Whole Blood",Blood,Blood Gas
31,50833,Potassium,Other Body Fluid,Blood Gas
45,50847,"Potassium, Ascites",Ascites,Chemistry
168,50971,Potassium,Blood,Chemistry
229,51041,"Potassium, Body Fluid",Other Body Fluid,Chemistry
245,51057,"Potassium, Pleural",Pleural,Chemistry
252,51064,"Potassium, Stool",Stool,Chemistry
285,51097,"Potassium, Urine",Urine,Chemistry
916,51800,"Potassium, CSF",Cerebrospinal Fluid,Chemistry
938,51822,"Potassium, Joint Fluid",Joint Fluid,Chemistry


Pick the potassium labels that reflect systemic electrolyte levels (blood as fluid).

In [80]:
# Grab potassium itemids:
potassium_ids = [50822, 50971, 52452, 52610]

In [81]:
# Bilirubin:
d_labitems[d_labitems["label"].str.contains("bilirubin", case=False, na=False)]

,itemid,label,fluid,category
36,50838,"Bilirubin, Total, Ascites",Ascites,Chemistry
81,50883,"Bilirubin, Direct",Blood,Chemistry
82,50884,"Bilirubin, Indirect",Blood,Chemistry
83,50885,"Bilirubin, Total",Blood,Chemistry
216,51028,"Bilirubin, Total, Body Fluid",Other Body Fluid,Chemistry
237,51049,"Bilirubin, Total, Pleural",Pleural,Chemistry
624,51464,Bilirubin,Urine,Hematology
625,51465,Bilirubin Crystals,Urine,Hematology
691,51568,"Bilirubin, Neonatal",Blood,Chemistry
692,51569,"Bilirubin, Neonatal, Direct",Blood,Chemistry


Pick total bilirubin (yellow pigment produced when red blood cells are broken down, usually an indicator of liver health).

In [82]:
# Grab bilirubin itemids:
bilirubin_ids = [50885, 53089]

In [83]:
# Build lab itemid map:
lab_itemids = {
    "lactate": [50813, 52442, 53154],
    "creatinine": [50912, 52024, 52546],
    "wbc": [51301, 51755, 51756],
    "platelets": [51265, 53189],
    "sodium": [50824, 50983, 52455, 52623],
    "potassium": [50822, 50971, 52452, 52610],
    "bilirubin": [50885, 53089]
}

# Flatten:
selected_lab_itemids = [50813, 52442, 53154, 50912, 52024, 
                        52546, 51301, 51755, 51756, 51265, 
                        53189, 50824, 50983, 52455, 52623, 
                        50822, 50971, 52452, 52610, 50885, 
                        53089]

# Verify that we selected the correct items before moving on:
d_labitems[d_labitems["itemid"].isin(selected_lab_itemids)][
    ["itemid", "label"]
]

,itemid,label
11,50813,Lactate
20,50822,"Potassium, Whole Blood"
22,50824,"Sodium, Whole Blood"
83,50885,"Bilirubin, Total"
110,50912,Creatinine
168,50971,Potassium
180,50983,Sodium
450,51265,Platelet Count
486,51301,White Blood Cells
872,51755,White Blood Cells


## **Join Time-Series Data:**

In [84]:
# Create a lookup table ('chartevents' uses stay_id, 'labevents' uses hadm_id):
window_lookup = cohort[["stay_id", "hadm_id", "intime", "window_24h_end"]].copy()

In [85]:
# Make variable maps (itemid to feature name):
vital_map = {}

for feature, itemids in vital_itemids.items():
    for itemid in itemids:
        vital_map[itemid] = feature

print(vital_map)

lab_map = {}

for feature, itemids in lab_itemids.items():
    for itemid in itemids:
        lab_map[itemid] = feature

print(lab_map)

{220045: 'heart_rate', 220052: 'map', 220181: 'map', 223762: 'temperature', 220210: 'resp_rate', 220277: 'spo2'}
{50813: 'lactate', 52442: 'lactate', 53154: 'lactate', 50912: 'creatinine', 52024: 'creatinine', 52546: 'creatinine', 51301: 'wbc', 51755: 'wbc', 51756: 'wbc', 51265: 'platelets', 53189: 'platelets', 50824: 'sodium', 50983: 'sodium', 52455: 'sodium', 52623: 'sodium', 50822: 'potassium', 50971: 'potassium', 52452: 'potassium', 52610: 'potassium', 50885: 'bilirubin', 53089: 'bilirubin'}


In [86]:
# Extract 24h vitals from chartevents in CHUNKS:
CHARTEVENTS_CSV = DATA_DIR / "icu" / "chartevents.csv"

stay_ids = set(cohort["stay_id"])
window_lookup_stay = window_lookup.set_index("stay_id")

chartevent_chunks = []
chunksize = 1_000_000

usecols_ce = ["stay_id", "charttime", "itemid", "valuenum"]

for chunk in pd.read_csv(
    CHARTEVENTS_CSV,
    usecols=usecols_ce,
    chunksize=chunksize,
    low_memory=False
):
    chunk = chunk[chunk["itemid"].isin(selected_itemids)]
    chunk = chunk[chunk["stay_id"].isin(stay_ids)]
    chunk = chunk.dropna(subset=["stay_id", "charttime", "valuenum"])

    if chunk.empty:
        continue

    chunk["charttime"] = pd.to_datetime(chunk["charttime"], errors="coerce")
    chunk = chunk.dropna(subset=["charttime"])

    chunk = chunk.merge(
        window_lookup[["stay_id", "intime", "window_24h_end"]],
        on="stay_id",
        how="inner"
    )

    chunk = chunk[
        (chunk["charttime"] >= chunk["intime"]) &
        (chunk["charttime"] <= chunk["window_24h_end"])
    ]

    if chunk.empty:
        continue

    chunk["feature"] = chunk["itemid"].map(vital_map)
    chunk = chunk.dropna(subset=["feature"])

    chartevent_chunks.append(
        chunk[["stay_id", "feature", "charttime", "valuenum"]]
    )

chartevents_24h = pd.concat(chartevent_chunks, ignore_index=True)

In [87]:
# Aggregate vitals:
vitals_agg = (
    chartevents_24h
    .sort_values(["stay_id", "feature", "charttime"])
    .groupby(["stay_id", "feature"])
    .agg(
        min_val=("valuenum", "min"),
        max_val=("valuenum", "max"),
        mean_val=("valuenum", "mean"),
        last_val=("valuenum", "last")
    )
    .reset_index()
)

In [88]:
# Pivot wide:
vitals_wide = vitals_agg.pivot(
    index="stay_id",
    columns="feature",
    values=["min_val", "max_val", "mean_val", "last_val"]
)

vitals_wide.columns = [
    f"{stat}_{feature}" for stat, feature in vitals_wide.columns
]

vitals_wide = vitals_wide.reset_index()

In [89]:
# Extract 24h labs from labevents in CHUNKS:
LABEVENTS_CSV = DATA_DIR / "hosp" / "labevents.csv"

hadm_ids = set(cohort["hadm_id"])

lab_chunks = []
chunksize = 1_000_000

usecols_le = ["hadm_id", "charttime", "itemid", "valuenum"]

for chunk in pd.read_csv(
    LABEVENTS_CSV,
    usecols=usecols_le,
    chunksize=chunksize,
    low_memory=False
):
    chunk = chunk[chunk["itemid"].isin(selected_lab_itemids)]
    chunk = chunk[chunk["hadm_id"].isin(hadm_ids)]
    chunk = chunk.dropna(subset=["hadm_id", "charttime", "valuenum"])

    if chunk.empty:
        continue

    chunk["charttime"] = pd.to_datetime(chunk["charttime"], errors="coerce")
    chunk = chunk.dropna(subset=["charttime"])

    chunk = chunk.merge(
        window_lookup[["stay_id", "hadm_id", "intime", "window_24h_end"]],
        on="hadm_id",
        how="inner"
    )

    chunk = chunk[
        (chunk["charttime"] >= chunk["intime"]) &
        (chunk["charttime"] <= chunk["window_24h_end"])
    ]

    if chunk.empty:
        continue

    chunk["feature"] = chunk["itemid"].map(lab_map)
    chunk = chunk.dropna(subset=["feature"])

    lab_chunks.append(
        chunk[["stay_id", "feature", "charttime", "valuenum"]]
    )

labevents_24h = pd.concat(lab_chunks, ignore_index=True)

In [90]:
# Aggregate labs:
labs_agg = (
    labevents_24h
    .sort_values(["stay_id", "feature", "charttime"])
    .groupby(["stay_id", "feature"])
    .agg(
        min_val=("valuenum", "min"),
        max_val=("valuenum", "max"),
        mean_val=("valuenum", "mean"),
        last_val=("valuenum", "last")
    )
    .reset_index()
)

In [91]:
# Pivot wide:
labs_wide = labs_agg.pivot(
    index="stay_id",
    columns="feature",
    values=["min_val", "max_val", "mean_val", "last_val"]
)

labs_wide.columns=[
    f"{stat}_{feature}" for stat, feature in labs_wide.columns
]

labs_wide = labs_wide.reset_index()

In [92]:
# Merge features back into cohort:
baseline_dataset = (
    cohort
    .merge(vitals_wide, on="stay_id", how="left")
    .merge(labs_wide, on="stay_id", how="left")
)

In [93]:
# Keep only patients that contribute to the ENTIRE 24-hour window:
baseline_dataset = baseline_dataset[
    baseline_dataset["ICU_time_to_event_hours"] >= 24
].copy()

In [94]:
# View missingness:
missingness = baseline_dataset.isna().mean().sort_values(ascending=False)
print(missingness)

min_val_temperature        0.906145
last_val_temperature       0.906145
mean_val_temperature       0.906145
max_val_temperature        0.906145
mean_val_bilirubin         0.548610
                             ...   
ICU_mortality              0.000000
ICU_endtime                0.000000
ICU_time_to_event_hours    0.000000
window_24h_end             0.000000
subject_id                 0.000000
Length: 66, dtype: float64


In [95]:
# Inspect feature coverage:
feature_cols = [c for c in baseline_dataset.columns if c.startswith(("min_val", "max_val", "mean_val", "last_val"))]
baseline_dataset[feature_cols].notna().mean().sort_values()

max_val_temperature     0.093855
mean_val_temperature    0.093855
min_val_temperature     0.093855
last_val_temperature    0.093855
mean_val_bilirubin      0.451390
min_val_bilirubin       0.451390
max_val_bilirubin       0.451390
last_val_bilirubin      0.451390
last_val_lactate        0.589947
mean_val_lactate        0.589947
max_val_lactate         0.589947
min_val_lactate         0.589947
mean_val_wbc            0.979553
max_val_wbc             0.979553
min_val_wbc             0.979553
last_val_wbc            0.979553
mean_val_platelets      0.979620
max_val_platelets       0.979620
last_val_platelets      0.979620
min_val_platelets       0.979620
min_val_creatinine      0.984447
max_val_creatinine      0.984447
last_val_creatinine     0.984447
mean_val_creatinine     0.984447
last_val_sodium         0.985399
min_val_sodium          0.985399
mean_val_sodium         0.985399
max_val_sodium          0.985399
max_val_potassium       0.985560
min_val_potassium       0.985560
last_val_p

In [96]:
# Preview:
baseline_dataset.head()

,subject_id,hadm_id,stay_id,first_careunit,intime,outtime,admission_type,admission_location,insurance,language,...,mean_val_potassium,mean_val_sodium,mean_val_wbc,last_val_bilirubin,last_val_creatinine,last_val_lactate,last_val_platelets,last_val_potassium,last_val_sodium,last_val_wbc
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,EW EMER.,EMERGENCY ROOM,Medicare,English,...,4.400000,137.000,7.500000,NaN,0.9,NaN,199.0,4.4,137.0,7.5
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,EW EMER.,EMERGENCY ROOM,Private,Other,...,3.600000,138.000,19.000000,NaN,0.4,NaN,285.0,3.6,138.0,19.0
5,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,EW EMER.,PACU,Private,English,...,3.700000,139.000,18.550000,NaN,0.8,NaN,299.0,3.5,138.0,20.1
7,10001884,26184834,37510196,Medical Intensive Care Unit (MICU),2131-01-11 04:20:05,2131-01-20 08:27:30,OBSERVATION ADMIT,EMERGENCY ROOM,Medicare,English,...,4.033333,136.000,15.200000,0.2,1.0,1.1,149.0,4.2,138.0,12.0
8,10002013,23581541,39060235,Cardiac Vascular Intensive Care Unit (CVICU),2160-05-18 10:00:53,2160-05-19 17:33:33,SURGICAL SAME DAY ADMISSION,PHYSICIAN REFERRAL,Medicare,English,...,3.980000,137.625,18.766667,NaN,0.9,2.6,248.0,4.5,136.0,17.9


## **Checkpoint:**

In [97]:
# Baseline 24h:
baseline_dataset.to_parquet(
    "/Users/gloriaduo/Documents/SJSU/SPRING 2026/CS 133 (Data Visualization)/Project/Predicting-Patient-Mortality/derived_results/baseline_dataset.parquet",
    index=False
)